# Historical exploration — archived, not a runnable release example
All saved outputs were cleared during the reproducibility review. The original code contains a personal import path, obsolete MISTIC arguments, and independently inferred feature grids.
Use `research/CPP_reproducible.ipynb` and `research/cpp_workflow.py` instead. The corrected workflow excludes four exact train/test duplicates and records pinned dependencies and data provenance. See `research/README.md` and `research/DATA_PROVENANCE.md`.
The original source is retained for provenance, not as a reproduction of earlier numerical claims.

In [ ]:
%load_ext autoreload

%autoreload 2

In [ ]:
import sys
sys.path.append('/Users/c.kieslich/Box Sync/Research/featureSelection/')

from mistic.svmSet import svmSet
from mistic.cvSet import cvSet 
from mistic.utility import combined_rank, kernelWrapper, score_svc, perDiff, paramSet

from FFTpro.features import SequenceSet, SequenceFFT 
from FFTpro.embeddings import b100

import matplotlib.pyplot as plt   
import numpy as np
import pandas as pd
import copy

from sklearn.preprocessing import StandardScaler, MinMaxScaler 
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC

In [ ]:
# Import peptide database
training_set = pd.read_csv("CPP_training.csv", sep = ",") 
testing_set = pd.read_csv("CPP_independent.csv", sep = ",") 

In [ ]:
training_fft = SequenceSet(embedding = b100)
training_fft.from_sequence_list(training_set.Sequence,
                                attributes = {'Class':training_set.Class,'ID':training_set.index})
training_feats = training_fft.export_datatable(spectra = ['spectra'], 
                                               attributes = ['Class','ID'],
                                               compute_phase = False)

testing_fft = SequenceSet(embedding = b100)
testing_fft.from_sequence_list(testing_set.Sequence,
                               attributes = {'Class':testing_set.Class,'ID':testing_set.index})

testing_feats = testing_fft.export_datatable(spectra = ['spectra'], 
                                             attributes = ['Class','ID'],
                                             compute_phase = False)

testing_feats

In [ ]:
# Scale input X
X = training_feats.copy()
X.drop(columns=['ID','Class'],inplace=True)
scaler = StandardScaler()
scaler.fit(X)

In [ ]:
X_train = scaler.transform(X)
y_train = training_feats.Class

X_val = testing_feats.copy()
X_val.drop(columns=['ID','Class'],inplace=True)
X_test = scaler.transform(X_val)
y_test = testing_feats.Class

In [ ]:
cv_set = cvSet(X = X_train, y = y_train.values)
cv_set.independent(num_sets = 3)

In [ ]:
kernel_type = "rbf"

C_vals = [2**x for x in range(1,9)]
gamma_vals = [2**x for x in range(-8,1)]

parameter_grid = []
for cost in C_vals:
    for gamma in gamma_vals:
        parameter_grid.append(paramSet(model={"C": cost},kernel={"gamma": gamma}))

svc = SVC(kernel = 'precomputed', class_weight="balanced", probability=False)

test = svmSet(svc, cv_set,
              score_method = score_svc().score,
              kernel = kernelWrapper(type = kernel_type),
              separate_parameters = True)

test.tune_models(parameter_grid)
test.performance_ 

In [ ]:
rank_weights = [0,0.25,0.50,0.75,1]

svm_set_ranks = []
for i in range(len(rank_weights)):
    print(f"Rank weight: {rank_weights[i]:.2f}")
    svm_set_ranks.append(svmSet(svc, cv_set,
                                score_method = score_svc().score,
                                kernel = kernelWrapper(type = kernel_type),
                                separate_feature_sets = False,
                                separate_parameters = False,
                                sparse_kernel_matrix = False))

    svm_set_ranks[i].greedy_backward_selection(parameter_grid = parameter_grid,
                                           reduction_factor = 0.1,
                                           feature_ranker = combined_rank(weight=rank_weights[i]).compute,
                                           set_for_rank = "sample")

In [ ]:
svm_set_ranks[0].performance_

In [ ]:
C_vals = [2**x for x in range(1,9)]
gamma_vals = [2**x for x in range(-8,1)]

parameter_grid = []
for cost in C_vals:
    for gamma in gamma_vals:
        parameter_grid.append(paramSet(model={"C": cost},kernel={"gamma": gamma}))

svm_set_ranks[0].tune_models(parameter_grid)
svm_set_ranks[0].performance_

In [ ]:
for i in range(len(rank_weights)):
    svm_set_ranks[i].plot_performance(metric="score")

plt.legend(labels=rank_weights)

In [ ]:
for i in range(len(rank_weights)):
    svm_set_ranks[i].plot_performance(metric="f1")

plt.legend(labels=rank_weights)

In [ ]:
for i in range(len(rank_weights)):
    svm_set_ranks[i].plot_performance(metric="auc")

plt.legend(labels=rank_weights)

In [ ]:
enrich_score = []
for i in range(len(rank_weights)):
    enrich_score.append(svm_set_ranks[i].enrichment_score(type='auc'))

selected_run = enrich_score.index(max(enrich_score))

enrich_score

In [ ]:
from sklearn.metrics import roc_auc_score, confusion_matrix

decision_values = svm_set_ranks[selected_run].decision_function(X_test)
y_pred =  svm_set_ranks[selected_run].predict(X_test, use_voting = True)
#svm_set_ranks[selected_run].predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
precision = tp/(tp+fp)
recall = tp/(tp+fn)
f1 = 2*precision*recall/(precision+recall)

auc = roc_auc_score(y_test, decision_values)
            
blind_results = {'rank_weights': rank_weights[selected_run], 'number_features': len(svm_set_ranks[selected_run].features), 'f1': f1, 'auc': auc}

blind_results

In [ ]:
from seaborn import pairplot, clustermap, load_dataset
from scipy.cluster.hierarchy import linkage, dendrogram, leaves_list, fcluster

linkMeth = "average"
selected_model = 0

IG = svm_set_ranks[selected_run].integrated_gradient(X_test, model_index = selected_model)

IGtable = pd.DataFrame(IG,columns=X.columns[svm_set_ranks[selected_run].features])

distMatCol = perDiff(IGtable)
hcCol = linkage(distMatCol, method=linkMeth,optimal_ordering=True)

tr_IGtable = IGtable.transpose()
distMatRow = perDiff(tr_IGtable)
hcRow = linkage(distMatRow, method=linkMeth,optimal_ordering=True)

In [ ]:
color_val = max(np.abs((np.mean(IG)-np.std(IG), np.mean(IG)+np.std(IG))))
clustermap(IGtable, 
           yticklabels=False,
           cmap="vlag", vmin=-color_val, vmax=color_val,
           row_linkage=hcRow, col_linkage=hcCol, 
           row_colors=[["red","blue"][int(l!="Positive")+0] for l in y_test])

In [ ]:
import plotly.graph_objects as go
import plotly.express as px
import numpy as np

features = svm_set_ranks[selected_run].features
df_X_test = pd.DataFrame(X_test[:,features],columns = IGtable.columns)
N = len(IGtable)

fig = go.Figure()
fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=800,
    height=800,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = "Integrated Gradient",
)

fig.update_xaxes(
    zeroline=True,
    zerolinecolor="black",
    )
    
fig.update_yaxes(
    zeroline=True,
    zerolinecolor="black",
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)

num_features = 20
feature_order = np.argsort(np.sum(abs(IG),axis=0))[(len(features)-num_features):(len(features))]

y_val = 1
for f in feature_order:
    fig.add_trace(go.Scatter(
        x = IGtable.iloc[:,f], 
        y = y_val + np.random.rand(N)*0.5 - 0.25,
        mode = 'markers',
        marker = dict(size=9,
                      color = df_X_test.iloc[:,f],
                      coloraxis = "coloraxis",
                     ),
        name=IGtable.columns[f],
    ))
    y_val += 1


fig.update_layout(showlegend=False, coloraxis_showscale=True)
fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text="Feature Value",side = "right"),
                    cmin = -1, cmax = 2)
fig.update_yaxes(tickvals=[i for i in range(1,len(feature_order)+1)], ticktext=IGtable.columns[feature_order])
fig.show() 

In [ ]:
feat_1 = "Amp_W_0.0625"
feat_2 = "Amp_R_0.5"

fig = px.scatter(x = df_X_test[feat_1], y = IGtable[feat_1], color = df_X_test[feat_2])
fig.update_traces(marker=dict(size=12))

fig.update_layout(
    plot_bgcolor='white',
    autosize=False,
    width=800,
    height=800,
    coloraxis = {'colorscale':'Bluered'},
    xaxis_title = feat_1,
    yaxis_title = "Integrated Gradient for " + feat_1,
)

fig.update_xaxes(
    zeroline=True,
    zerolinecolor='lightgrey',
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
    )
    
fig.update_yaxes(
    zeroline=True,
    zerolinecolor='lightgrey',
    mirror=True,
    ticks='outside',
    showline=True,
    linecolor='black',
    gridcolor='lightgrey'
)

fig.update_coloraxes(colorbar_showticklabels=False,
                    colorbar_title=dict(text=feat_2,side = "right"),
                    cmin = -1, cmax = 2)

fig.show()